# 单细胞转座酶可及染色质测序（scATAC-seq）

本次分析使用 Kumegawa 等人在 2022 年发表的研究数据，论文题为 ["GRHL2 motif is associated with intratumor heterogeneity of cis-regulatory elements in luminal breast cancer"](https://pmc.ncbi.nlm.nih.gov/articles/PMC9177858/)。

在这篇论文中，Kumegawa 等人分析了 16 位乳腺癌患者中超过 10,000 个细胞的染色质可及性图谱，患者亚型包括 luminal、luminal-HER2、HER2+ 以及 3 种 triple-negative breast cancer（TNBC）亚型。基于这些图谱，作者将细胞分为癌细胞和肿瘤微环境细胞，从而揭示了疾病相关通路的异质性。此外，他们发现转录因子 GRHL2 可与 FOXA1 协同启动内分泌耐药，并且 GRHL2 结合元件可能调控与内分泌耐药、转移以及接受激素治疗患者预后不良相关的基因。

scATAC-seq 文库使用 SureCell ATAC-seq Library Preparation Kit（Bio-Rad）和 SureCell ddSEQ Index Kit（Bio-Rad）制备；比对步骤使用 ATAC-Seq Analysis Toolkit（Bio-Rad）完成。

本教程将探索两个乳腺肿瘤样本（TNBC 和 Luminal-HER2），并具体关注其中的 T 细胞。为此，我们根据作者提供的 GEO 编号 GSE198639，从 [GEO（Gene Expression Omnibus）网站](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi) 获取样本的 fragment 文件。

本课程使用 ArchR 完成。ArchR 的更多信息见[官方文档](https://www.archrproject.com/index.html)。

ArchR 为 scATAC-seq 分析提供了从预处理到结果解释的完整工具集，可以在多个层次上提取信息。同时，ArchR 运行速度较快，对计算资源的需求也相对合理。

如果要在自己的电脑上运行这些分析，需要完成以下准备：

1. 安装 Python 3.6 或更高版本：

https://www.python.org/downloads/

2. 安装 conda（miniconda 或 anaconda）。conda 是 Python 的包管理器，也可以用来创建独立的 Python 环境：

https://conda.io/projects/conda/en/latest/user-guide/install/index.html

3. 在终端中安装 macs2：

`conda create -y -n MACS2 python=3.6`

`conda activate MACS2`

`conda install macs2` 或 `conda install -c bioconda macs2`

* 安装 R 4.1.3 或更高版本：

https://cran.r-project.org/

* 在 R 环境中安装以下 R 包：

`install.packages(c("devtools","BiocManager","reticulate","clustree","Seurat"))`

`devtools::install_github("GreenleafLab/ArchR", ref="master", repos = BiocManager::repositories())`

`ArchR::installExtraPackages()`

`BiocManager::install("BSgenome.Hsapiens.UCSC.hg19")`（如果你的数据来自其他物种或其他基因组参考版本，应安装相应的 genome package）

`devtools::install_github("GreenleafLab/chromVARmotifs")`

`install.packages("hexbin")`


# 下载预安装库和数据集


In [1]:
# Download the installation script from GitHub
download.file("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_jammy.sh",
              "add_cranapt_jammy.sh")

# Change the script's permissions to make it executable
Sys.chmod("add_cranapt_jammy.sh", "0755")

# Execute the script in the terminal
system("./add_cranapt_jammy.sh")

In [2]:
system("apt install libfreetype6-dev libpng-dev libtiff5-dev libjpeg-dev libbz2-dev libgsl-dev gsl-bin -y")
system("apt install  libfontconfig1-dev libharfbuzz-dev libfribidi-dev libcairo2-dev libgmp-dev -y")
system("apt update")
system("apt install libmagick++-dev -y")

In [ ]:
# Define a helper function to run shell commands and print their output
shell_call <- function(command, ...) {
  result <- system(command, intern = TRUE, ...)  # Execute the command and capture output
  cat(paste0(result, collapse = "\n"))  # Print output to console
}

# Download the MACS2 package (version 2.2.9.1) using wget
shell_call("wget https://github.com/macs3-project/MACS/archive/refs/tags/v2.2.9.1.tar.gz -O MACS.tar.gz")

# Extract the downloaded tar.gz file
system("tar -xvf MACS.tar.gz")

# Install MACS2 in editable mode using pip
shell_call("pip install -e MACS-2.2.9.1/")

In [ ]:
# Set a long timeout limit to avoid download failures
options(timeout = 1000)

# Install BiocManager if not already installed
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager", quiet = TRUE)

# Install ArchR from GitHub using devtools
devtools::install_github("GreenleafLab/ArchR", ref = "master", repos = BiocManager::repositories(), upgrade = FALSE)

# Install additional dependencies for ArchR
ArchR::installExtraPackages()

# Install other required R packages
install.packages("clustree", quiet = TRUE)
install.packages("hexbin")

# Install a specific version of the Matrix package from CRAN archives
install.packages("https://cran.r-project.org/src/contrib/Archive/Matrix/Matrix_1.5-3.tar.gz", repos = NULL, type = "source")

In [1]:
# Function to run shell commands and display the output
shell_call <- function(command, ...) {
  result <- system(command, intern = TRUE, ...)  # Run command and store output
  cat(paste0(result, collapse = "\n"))  # Print the output
}

In [ ]:
# Set a timeout limit for downloads
options(timeout = 300)

# Download the scATAC-seq workshop dataset as a ZIP file
download.file('https://iauchile-my.sharepoint.com/:u:/g/personal/adolfo_rh_postqyf_uchile_cl/ETPOTjhE9llEkT85F6XQfyQBdN4r9R2Jf4hvY1BicfTWSw?e=tQbDjt&download=1', 
              'scATACseqWorkshop.zip')

# List the downloaded files with detailed information
shell_call("ls -lh")

# Unzip the downloaded file
system("unzip scATACseqWorkshop.zip")

# 注意：如果运行中出错，可以用下面的方法重新分析数据


In [ ]:
# Define the working directory
work_dir2 <- "/content/"
setwd(work_dir2)

# Remove any existing directory with the same name
shell_call("rm -rf scATACseqWorkshop/")

# Unzip the dataset again
shell_call("unzip scATACseqWorkshop.zip")

## 1. 定义库、参数和目录

首先，我们定义 Python 库的位置，并加载 R 包。随后设置一些分析参数，包括：1）使用的线程数，2）工作目录，3）fragment 文件的位置。

实际上，ArchR 支持多种 scATAC-seq 数据输入格式，其中 fragment 文件和 BAM 文件是最常见的 scATAC-seq 输入数据。


In [ ]:
# Suppress package startup messages for cleaner output
# Load libraries
suppressPackageStartupMessages({
  library(ArchR)
  library(reticulate)
  library(clustree)
  library(Seurat)
  library(hexbin)
})

In [ ]:
# Set the Python and install Macs2
system("python3 -m pip install macs2")
system("python3 -m pip show macs2 | grep Location")
writeLines("#!/usr/bin/env bash\npython3 -m macs2 \"$@\"", "/usr/local/bin/macs2")
system("chmod +x /usr/local/bin/macs2"

# Check the Python configuration
system("macs2 --version")

# Test if MACS2 is installed and accessible
findMacs2()

In [ ]:
# Set a random seed for reproducibility
set.seed(1)

# Define the number of threads to use
nb.threads = 2
addArchRThreads(threads = nb.threads)

# Set the working directory
work_dir <- "/content/scATACseqWorkshop"
# Create if it doesn't exist
dir.create(work_dir, showWarnings = FALSE)
setwd(work_dir)

# List and name the input fragment files
inputFiles <- list.files(file.path(work_dir, "fragments_data"), full.names = TRUE)
names(inputFiles) <- gsub("^.+/", "", gsub("GSM[0-9]+_", "", gsub(".fragments.tsv.gz", "", inputFiles)))

# Specify the reference genome for ArchR
addArchRGenome("hg19")

## 2. 创建 Arrow 文件

我们会创建 HDF5 格式的 Arrow 文件，用于存储与样本相关的所有数据。该文件会贯穿整个分析过程，并随着分析推进不断加入新的信息层。

如果分析多个样本，每个样本都会生成一个对应的 Arrow 文件。

Arrow 文件本身不是 R 语言对象。因此，我们还会生成一个 ArchRProject 对象，把一个或多个 Arrow 文件纳入同一个分析框架，使其能够在 R 中被快速访问和操作。

在这一步中，ArchR 会计算 TileMatrix，其中包含全基因组 500 bp bins（默认值）上的 insertion counts；同时还会计算 GeneScoreMatrix，该矩阵根据基因启动子附近 tiles 中的 insertion counts 加权估计基因表达。


In [ ]:
inputFiles

In [ ]:
reformatFragmentFiles(inputFiles)

In [ ]:
inputFiles <- gsub("fragments.tsv.gz","fragments-Reformat.tsv.gz",inputFiles)

In [ ]:
ArrowFiles <- createArrowFiles(
  inputFiles = inputFiles,   # Input files containing scATAC-seq data
  sampleNames = names(inputFiles),  # Assign sample names based on input file names
  minTSS = 0.1,   # Minimum TSS enrichment score to filter low-quality cells
  minFrags = 1,   # Minimum number of unique fragments per cell
  addTileMat = TRUE,  # Compute and store the tile matrix for accessibility analysis
  addGeneScoreMat = TRUE  # Compute and store the gene score matrix for gene activity analysis
)

# Create an ArchR project using the Arrow files
project <- ArchRProject(
  ArrowFiles = ArrowFiles,  # Use the generated Arrow files
  outputDirectory = "Analysis_scATACseq_noFilter",  # Define output directory for the project
  copyArrows = TRUE  # Recommended to maintain an unaltered copy of Arrow files for future use
)

> 如果由于任何原因 Arrow 文件创建失败，或者运行时间过长，可以从下面的 GitHub 链接下载预先生成好的版本。


In [ ]:
#download.file('https://github.com/integrativebioinformatics/scNotebooks/blob/main/scNotebooks-Resources/P41.arrow','P41.arrow')
#download.file('https://github.com/integrativebioinformatics/scNotebooks/blob/main/scNotebooks-Resources/P93.arrow','P93.arrow')
#ArrowFiles <- c('P41.arrow','P93.arrow')

对 scATAC-seq 数据进行严格质量控制（quality control, QC）非常重要，可以去除低质量细胞对分析结果的影响。

ArchR 主要考察数据的三个特征：

1. fragment size 分布。fragment 是由 Tn5 转座酶切割得到的 DNA 片段。由于核小体周期性，我们通常期望看到长度约为 147 bp，也就是缠绕在一个核小体上的 DNA 长度附近的片段减少。

2. Transcription Start Site（TSS）enrichment，即信噪比。较低的信噪比通常与死亡或濒死细胞有关，这类细胞的 DNA 去染色质化后更容易在全基因组范围内发生随机转座。

3. unique nuclear fragments 数量，即不比对到线粒体 DNA 的唯一核基因组 fragment 数量。

可以通过一些图形查看样本的 QC 情况和主要指标。

绘制 QC 指标：


In [ ]:
# Extract TSS enrichment and fragment count data from the ArchR project
df <- getCellColData(project, select = c("log10(nFrags)", "TSSEnrichment"))

# Create a scatter plot of TSS Enrichment vs. Log10(Unique Fragments)
plot.tss.frags <- ggPoint(
  x = df[,1], y = df[,2],  # Set x-axis as log10(Unique Fragments) and y-axis as TSS Enrichment
  colorDensity = TRUE,  # Color points based on density
  continuousSet = "sambaNight",  # Define color theme
  xlabel = "Log10 Unique Fragments", ylabel = "TSS Enrichment",  # Label axes
  xlim = c(0, quantile(df[,1], probs = 1) + 0.1),  # Set x-axis limits
  ylim = c(0, quantile(df[,2], probs = 1) + 0.1)   # Set y-axis limits
)

# Save the plot as a PDF in the project's "Plots" directory
plotPDF(plot.tss.frags, name = "TSS-vs-Frags.pdf", ArchRProj = project, addDOC = FALSE)

# Display the plot
plot.tss.frags

绘制 TSS 指标：


In [ ]:
# Create a ridge plot showing TSS enrichment distribution across samples
plot.tss.v1 <- plotGroups(
  ArchRProj = project,
  groupBy = "Sample",  # Group by sample
  colorBy = "cellColData",  # Color based on cell metadata
  name = "TSSEnrichment",  # Use TSS Enrichment as the feature to plot
  plotAs = "ridges"  # Plot as ridge plot
)

# Create a violin plot with an overlaid box plot
plot.tss.v2 <- plotGroups(
  ArchRProj = project,
  groupBy = "Sample",
  colorBy = "cellColData",
  name = "TSSEnrichment",
  plotAs = "violin",  # Plot as a violin plot
  alpha = 0.4,  # Set transparency level
  addBoxPlot = TRUE  # Overlay a box plot on top of the violin plot
)

# Display both plots side by side
plot.tss.v1 | plot.tss.v2

绘制 fragment 指标：


In [ ]:
# Create a ridge plot showing log10(Unique Fragments) distribution across samples
plot.frags.v1 <- plotGroups(
  ArchRProj = project,
  groupBy = "Sample",
  colorBy = "cellColData",
  name = "log10(nFrags)",
  plotAs = "ridges"
)

# Create a violin plot with an overlaid box plot
plot.frags.v2 <- plotGroups(
  ArchRProj = project,
  groupBy = "Sample",
  colorBy = "cellColData",
  name = "log10(nFrags)",
  plotAs = "violin",
  alpha = 0.4,
  addBoxPlot = TRUE
)

# Display both plots side by side
plot.frags.v1 | plot.frags.v2

In [ ]:
# Create Arrow Files with Quality Filters
ArrowFiles <- createArrowFiles(
  inputFiles = inputFiles,  # List of fragment files for each sample
  sampleNames = names(inputFiles),  # Assign sample names based on file names
  minTSS = 4,  # Minimum Transcription Start Site (TSS) enrichment score to retain a cell
  minFrags = 1000,  # Minimum number of unique fragments per cell
  addTileMat = TRUE,  # Create a tile matrix for peak calling and other analyses
  addGeneScoreMat = TRUE  # Compute gene activity scores
)

## 3. Doublet 检测

单细胞数据中的一个常见问题是 doublets 对分析的影响。doublet 指同一个液滴中捕获了不止一个细胞核。

为了预测哪些“细胞”实际上是 doublets，ArchR 会从数据中合成 in silico doublets：它将许多单细胞组合的 reads 混合起来，构造出模拟 doublet。随后，ArchR 会把这些合成 doublets 投影到 UMAP embedding 中并寻找其最近邻。通过重复这一过程数千次，ArchR 可以识别真实数据中信号模式与合成 doublets 非常相似的“细胞”。

这里，我们将识别 doublets。


In [ ]:
# Compute Doublet Scores
doubletScores <- addDoubletScores(
  input = ArrowFiles,  # Use Arrow files created in the previous step
  k = 10,  # Number of nearest neighbors to consider for doublet detection
  knnMethod = "UMAP",  # Use UMAP embedding for nearest neighbor search
  LSIMethod = 1  # Use Latent Semantic Indexing (LSI) method 1 for doublet estimation
)

## 4. 创建 ArchR project

如前所述，我们会生成一个 ArchR project，以便更方便地操作由 ArchR 创建的 scATAC-seq 数据。


In [ ]:
# Create an ArchR Project
project <- ArchRProject(
  ArrowFiles = ArrowFiles,  # Use the Arrow files created earlier
  outputDirectory = "Analysis_scATACseq",  # Define output directory for the project
  copyArrows = TRUE  # Keep a copy of the Arrow files for future reference
)

我们可以很方便地列出 project 中包含的矩阵项目。


In [ ]:
# List Available Matrices in the Project
getAvailableMatrices(project)  # Check what matrices (e.g., Gene Score Matrix) are available

对 scATAC-seq 数据进行严格质量控制（QC）非常重要，可以去除低质量细胞对分析结果的影响。

ArchR 主要考察数据的三个特征：

1. fragment size 分布。由于核小体周期性，我们通常期望看到长度约为 147 bp，也就是缠绕在一个核小体上的 DNA 长度附近的片段减少。

2. TSS enrichment，即信噪比。较低的信噪比通常与死亡或濒死细胞有关，这类细胞的 DNA 去染色质化后更容易在全基因组范围内发生随机转座。

3. unique nuclear fragments 数量，即不比对到线粒体 DNA 的唯一核基因组 fragment 数量。

可以通过一些图形查看样本的 QC 情况和主要指标。

绘制 QC 指标：


In [ ]:
# Plot TSS Enrichment vs. Fragment Counts
df <- getCellColData(project, select = c("log10(nFrags)", "TSSEnrichment"))  # Extract metadata

plot.tss.frags <- ggPoint(
  x = df[,1],  # Log10 number of unique fragments
  y = df[,2],  # TSS Enrichment score
  colorDensity = TRUE,  # Color by density
  continuousSet = "sambaNight",  # Use the "sambaNight" color scheme
  xlabel = "Log10 Unique Fragments",  # Label for X-axis
  ylabel = "TSS Enrichment",  # Label for Y-axis
  xlim = c(log10(450), quantile(df[,1], probs = 1) + 0.1),  # Set X-axis limits
  ylim = c(0, quantile(df[,2], probs = 1) + 0.1)  # Set Y-axis limits
) +
  geom_hline(yintercept = 4, lty = "dashed", col = "black") +  # Add horizontal line at TSS = 4
  geom_vline(xintercept = log10(1000), lty = "dashed", col = "black")  # Add vertical line at 1000 fragments

# Save the plot as a PDF inside the project directory
plotPDF(plot.tss.frags, name = "TSS-vs-Frags.pdf", ArchRProj = project, addDOC = FALSE)

# Display the plot
plot.tss.frags

绘制 TSS 指标：


In [ ]:
# Plot TSS Enrichment Distributions

# Ridge plot of TSS enrichment per sample
plot.tss.v1 <- plotGroups(
  ArchRProj = project, 
  groupBy = "Sample",  # Group by sample
  colorBy = "cellColData",  # Use cell metadata for color
  name = "TSSEnrichment",  # Plot TSS enrichment scores
  plotAs = "ridges"  # Use a ridge plot
)

# Violin plot of TSS enrichment per sample with an overlaid boxplot
plot.tss.v2 <- plotGroups(
  ArchRProj = project, 
  groupBy = "Sample",  # Group by sample
  colorBy = "cellColData",  # Use cell metadata for color
  name = "TSSEnrichment",  # Plot TSS enrichment scores
  plotAs = "violin",  # Use a violin plot
  alpha = 0.4,  # Set transparency
  addBoxPlot = TRUE  # Add a boxplot overlay
)

# Display the ridge and violin plots side by side
plot.tss.v1 | plot.tss.v2

绘制 fragment 指标：


In [ ]:
# Plot Fragment Count Distributions

# Ridge plot of log10 fragment counts per sample
plot.frags.v1 <- plotGroups(
  ArchRProj = project, 
  groupBy = "Sample",  # Group by sample
  colorBy = "cellColData",  # Use cell metadata for color
  name = "log10(nFrags)",  # Plot log10 fragment counts
  plotAs = "ridges"  # Use a ridge plot
)

# Violin plot of log10 fragment counts per sample with an overlaid boxplot
plot.frags.v2 <- plotGroups(
  ArchRProj = project, 
  groupBy = "Sample",  # Group by sample
  colorBy = "cellColData",  # Use cell metadata for color
  name = "log10(nFrags)",  # Plot log10 fragment counts
  plotAs = "violin",  # Use a violin plot
  alpha = 0.4,  # Set transparency
  addBoxPlot = TRUE  # Add a boxplot overlay
)

# Display the ridge and violin plots side by side
plot.frags.v1 | plot.frags.v2

过滤 doublets。


In [ ]:
# Filter Out Doublets from the Dataset
project <- filterDoublets(ArchRProj = project)  # Remove detected doublets to keep only single cells


样本 fragment size 分布和 TSS enrichment profiles。


In [ ]:
# Plot the Transcription Start Site (TSS) enrichment profile
plot.tss.v3 <- plotTSSEnrichment(ArchRProj = project)

# Plot the fragment size distribution
plot.frags.v3 <- plotFragmentSizes(ArchRProj = project)

# Display both plots side by side
plot.frags.v3 | plot.tss.v3

## 5. 归一化、降维、批次效应校正、聚类及其他步骤


### 5.1. 归一化与降维

scATAC-seq 会生成非常稀疏的 insertion counts 矩阵，例如 500 bp tiles，对应约 600 万个二值特征。这使得使用标准降维方法识别高变 peaks 变得不可行。为了解决这个问题，ArchR 使用 LSI（latent semantic indexing），这是一种适用于稀疏、噪声较高数据的分层降维方法。

ArchR 并不直接寻找变异最大的 peaks，而是尝试使用最可及的特征作为 LSI 输入。

不过，在同时分析多个样本时，结果可能出现较高噪声且可重复性较低。

为了解决这一问题，ArchR 引入了 “iterative LSI” 方法（Satpathy, Granja et al., 2019）。该方法先在最可及的 tiles 上计算初始 LSI 转换，并识别较低分辨率、且不被批次混杂驱动的 clusters。

1. 首先，在最可及的 tiles 上计算初始 LSI 转换，并识别较低分辨率且不受批次混杂影响的 clusters。

2. 随后，ArchR 会计算这些 clusters 在所有特征上的平均可及性，并识别 clusters 之间变异最大的 peaks，再用这些特征重新进行 LSI。

3. 在第二轮迭代中，最具变异性的 peaks 更类似于 scRNA-seq LSI 实现中使用的高变基因。

这种策略可以降低观察到的批次效应，并把降维操作限制在规模更合理的特征矩阵上。

![](./Figures/iLSI.png)


In [ ]:
project_Normalized <- addIterativeLSI(ArchRProj = project, iterations = 2,
                                      # Number of iterations for LSI; more iterations refine clustering
                                      #sampleCellsPre = 50000, # Optional: Number of cells to use for iterations before the final one
                                      #clusterParams = list(resolution = 0.1, sampleCells = 50000, maxClusters = 6, n.start = 10), 
                                      # Cluster parameters can be adjusted to optimize clustering
                                      useMatrix = "TileMatrix", # Use TileMatrix for LSI
                                      name = "IterativeLSI", # Name of the reduced dimensions
                                      varFeatures = 25000) # Number of variable features to use for LSI

### 5.2. 批次效应校正

有时 iterative LSI 仍不足以校正较强的批次差异。因此，ArchR 实现了常用的批次效应校正工具 Harmony（Korsunsky et al., 2019）。Harmony 最初是为 scRNA-seq 设计的。


In [ ]:
# Perform batch correction using Harmony on the reduced dimensions from LSI
project_Normalized <- addHarmony(ArchRProj = project_Normalized, reducedDims = "IterativeLSI",
                                 name = "Harmony", groupBy = "Sample")

### 5.3. UMAP

在 ArchR 中运行 UMAP。


In [ ]:
# Compute UMAP embedding based on Iterative LSI dimensions
project_Normalized <- addUMAP(ArchRProj = project_Normalized, reducedDims = "IterativeLSI", name = "UMAP")

# Plot UMAP colored by sample identity
plotEmbedding(ArchRProj = project_Normalized, colorBy = "cellColData", name = "Sample", embedding = "UMAP", size=0.1)

# Compute UMAP embedding based on Harmony dimensions (after batch correction)
project_Normalized <- addUMAP(ArchRProj = project_Normalized, reducedDims = "Harmony", name = "UMAP", force=TRUE)

# Plot UMAP again to visualize batch-corrected embedding
plotEmbedding(ArchRProj = project_Normalized, colorBy = "cellColData", name = "Sample", embedding = "UMAP", size=0.1)

### 5.4. 聚类

为了识别 clusters，ArchR 可以使用与 Seurat 或 Scran 相同的方法。这里选择的是第一天课程中介绍并在 Seurat 包中使用过的方法。


In [ ]:
# Iterate over different clustering resolutions from 0.0 to 0.9
for(i in seq(0,0.9,0.1)){
  project_Normalized <- addClusters(input = project_Normalized, reducedDims = "Harmony",
                                    method = "Seurat", # Clustering method (Seurat-based)
                                    name = paste("Clusters.res",i,sep=""), # Naming clusters dynamically
                                    resolution = i, # Set clustering resolution
                                    verbose = FALSE) # Suppress verbose output
}

### 5.5. 保存和加载 project

保存：


In [ ]:
# Save the current state of the ArchR project to disk
saveArchRProject(ArchRProj = project_Normalized,
                 outputDirectory = file.path(getwd(),"Analysis_scATACseq"))

加载：


In [ ]:
# Load the saved ArchR project
project_Normalized <- loadArchRProject(path = file.path(getwd(),"Analysis_scATACseq"),
                                       force = TRUE, showLogo = FALSE)

## 6. 使用 GeneScore 估计探索数据


### 6.1. 使用 Clustree 可视化聚类

Clustree 是一个有用工具，可用于探索不同分辨率下 clusters 之间的对应关系。


In [ ]:
# Extract clustering information from ArchR project
tmp.clustree.datatable <- as.data.frame(project_Normalized@cellColData)

# Plot a clustering tree to visualize clustering relationships across resolutions
clustree(tmp.clustree.datatable, prefix="Clusters.res")

### 6.2. 在 UMAP 上可视化聚类


In [ ]:
# Iterate over different clustering resolutions and visualize UMAP embeddings
for(i in seq(0,0.9,0.1)){
  # Plot UMAP with cluster labels
  plot.umap.resi <- plotEmbedding(ArchRProj = project_Normalized, 
                                  colorBy = "cellColData", 
                                  name = paste("Clusters.res",i,sep=""), 
                                  embedding = "UMAP", size=0.1)
  
  # Plot UMAP without cluster labels
  plot.umap.woLabel.resi <- plotEmbedding(ArchRProj = project_Normalized, 
                                          colorBy = "cellColData", 
                                          name = paste("Clusters.res",i,sep=""), 
                                          embedding = "UMAP", size=0.1, 
                                          labelMeans=FALSE)

  # Display both plots side by side
  print(plot.umap.resi | plot.umap.woLabel.resi)
}

### 6.3. 表征 clusters

在这一步中，我们选择一个特定分辨率，并详细探索 GeneScore，用于刻画不同 clusters 的特征。

为此，我们会识别各 cluster 的 marker genes。这些 marker 基于 gene scores，也就是对基因表达的估计。

简而言之，ArchR 会基于基因区域的局部可及性估计 gene scores。该区域包括启动子和 gene body；同时，ArchR 会根据距离对潜在远端调控元件的活性施加指数权重。

说明：ArchR 可以使用 gene、peak 或 transcription factor motif 作为特征。这里，ArchR 会在分辨率 0.4 下识别每个 cluster 中看起来特异活跃的基因。

![<i><font size=1 color="grey">from ArchR manual</font></i>](./Figures/GeneActivityScore_Schematic.png){width=70% height=50%}


In [ ]:
slct.res = "res0.7" # Select resolution for analysis

# Identify marker genes using Gene Score Matrix
markersGS.slctRes <- getMarkerFeatures(ArchRProj = project_Normalized,
                                       useMatrix = "GeneScoreMatrix",
                                       groupBy = paste("Clusters.",slct.res,sep=""),
                                       bias = c("TSSEnrichment", "log10(nFrags)"),
                                       testMethod = "wilcoxon") # Perform Wilcoxon test

# Extract marker genes with FDR ≤ 0.05 and Log2 Fold Change ≥ 0.2
markerList <- getMarkers(markersGS.slctRes, cutOff = "FDR <= 0.05 & Log2FC >= 0.2")

# Display marker genes for the first cluster
i = names(markerList)[1]
markerList[[i]]

# Save marker genes for each cluster
for(i in names(markerList)){
  write.table(markerList[[i]], sep="\t", row.names=FALSE, col.names=TRUE, quote=FALSE,
              file=file.path(work_dir, paste(i, ".res", slct.res, ".mGenesList.tsv", sep="")))
}

为了可视化 marker genes，可以绘制热图：


In [ ]:
# Define key marker genes
markerGenes <- c("EPCAM", "VIM", "FLT4", "THY1", "CD3D", "PECAM1", "CD38", "PAX5",
                 "MS4A1", "CD14", "ITGAX", "CD4", "CD8A", "GZMA")

# Generate heatmap of gene scores
heatmapGS <- plotMarkerHeatmap(seMarker = markersGS.slctRes,
                               cutOff = "FDR <= 0.05 & Log2FC >= 1",
                               labelMarkers = markerGenes,
                               transpose = FALSE)

# Display heatmap
heatmapGS

# Retrieve heatmap matrix
heatmapGSmatrix <- plotMarkerHeatmap(seMarker = markersGS.slctRes,
                                     cutOff = "FDR <= 0.05 & Log2FC >= 1",
                                     labelMarkers = markerGenes,
                                     returnMatrix = TRUE,
                                     transpose = FALSE)

# Display first 10 rows of heatmap matrix
head(heatmapGSmatrix, 10)

# Save heatmap matrix
write.table(cbind(Cluster=rownames(heatmapGSmatrix), heatmapGSmatrix), sep="\t",
            row.names=FALSE, col.names=TRUE, quote=FALSE,
            file=file.path(work_dir, paste("GeneScores-Marker-Heatmap", slct.res, sep=".")))

也可以在 UMAP 上可视化 marker genes 的 GeneScore。


In [ ]:
# Plot Gene Score UMAP without MAGIC imputation
plot.GS.woMAGIC <- plotEmbedding(ArchRProj = project_Normalized, 
                                 colorBy = "GeneScoreMatrix", 
                                 name = markerGenes, embedding = "UMAP", 
                                 quantCut = c(0.01, 0.95), 
                                 imputeWeights = NULL)

# Display selected genes
plot.GS.woMAGIC$VIM | plot.GS.woMAGIC$EPCAM

不过，scATAC-seq 数据非常稀疏。因此，强烈建议使用 MAGIC（van Dijk et al., 2018），它通过平滑邻近细胞之间的信号，为 gene scores 添加 imputation weight。


In [ ]:
# Apply MAGIC for gene imputation
project_Normalized <- addImputeWeights(project_Normalized)

# Plot Gene Score UMAP with imputation
plot.GS <- plotEmbedding(ArchRProj = project_Normalized, colorBy = "GeneScoreMatrix",
                         name = markerGenes,
                         embedding = "UMAP",
                         imputeWeights = getImputeWeights(project_Normalized))

plot.GS$VIM | plot.GS$EPCAM
plot.GS$FLT4 | plot.GS$THY1
plot.GS$ITGAX | plot.GS$CD14
plot.GS$MS4A1 | plot.GS$CD38
plot.GS$CD3D | plot.GS$CD8A
plot.GS$CD4 | plot.GS$GZMA

## 7. scATAC 与 scRNA-seq 整合

ArchR 支持与 scRNA-seq 数据整合，并可以使用 scRNA-seq 空间中定义的 clusters，或使用整合后得到的基因表达测量结果。

这种整合的基本思路，是通过比较 scATAC-seq 的 gene score matrix 与 scRNA-seq 的 gene expression matrix，直接对齐 scATAC-seq 细胞和 scRNA-seq 细胞。该对齐过程使用 Seurat 包中的 `FindTransferAnchors()` 函数完成，它可以在两个数据集之间建立对应关系。

不过，为了让这一流程能够适用于数十万细胞规模的数据，ArchR 将所有细胞划分为更小的细胞组，并行执行多次独立对齐，从而实现并行化扩展。


In [31]:
# Import scRNAseq data
scRNA<-readRDS(file.path(work_dir,"scRNAseq.data.rds"))
DefaultAssay(object = scRNA) <- "RNA"

# Integrate scRNA-seq and scATAC-seq data
project_Normalized <- addGeneIntegrationMatrix(ArchRProj = project_Normalized,
    useMatrix = "GeneScoreMatrix", matrixName = "GeneIntegrationMatrix",
    reducedDims = "Harmony", #Harmony , IterativeLSI
    seRNA = scRNA, addToArrow = TRUE, force= TRUE,
    groupRNA = "integrated_snn_res.0.5",
    nameCell = "predictedCell", nameGroup = "predictedGroup", nameScore = "predictedScore",
    sampleCellsATAC = 10000, sampleCellsRNA = 10000, scaleTo = 10000)
project_Normalized <- addImputeWeights(project_Normalized)

saveArchRProject(ArchRProj = project_Normalized, load = FALSE)

plot_rna.woLabel <- plotEmbedding(project_Normalized,colorBy = "cellColData",name = "predictedGroup", embedding = "UMAP", size=1, labelMeans=FALSE)
plot_rna <- plotEmbedding(project_Normalized,colorBy = "cellColData",name = "predictedGroup", embedding = "UMAP", size=1)

# Cross table between scRNA-seq and scATAC-seq data
cM <- as.matrix(confusionMatrix(project$Clusters.res0.7,                # Warning resolution
                                project$predictedGroup))                # Warning resolution


## 8. Peak calling

Peak calling 是 ATAC-seq 数据分析中最基础的流程之一。

由于单细胞层面的 scATAC-seq 数据本质上接近二值，即某个位点可及或不可及，因此通常不会在单个细胞上进行 peak calling，而是在先前定义的相似细胞群或 clusters 上进行。

ArchR 使用推荐的 MACS2 peak caller（Zhang et al., 2008），并应用 iterative overlap peak merging procedure。

该过程通过函数执行，主要步骤如下：

1. ArchR 会分别为每个 pseudo-bulk replicate 进行 peak calling。

2. ArchR 会把来自同一细胞类型的所有 pseudo-bulk replicates 放在一起分析，并执行第一轮 iterative overlap removal。

3. 第一轮 iterative overlap removal 之后，ArchR 会检查每个 peak 在 pseudo-bulk replicates 之间的可重复性，只保留通过 reproducibility 参数阈值的 peaks。

4. 该过程结束后，每种细胞类型都会得到一个合并后的 peak set。


### 8.1. 创建 pseudo-bulk replicates


In [ ]:
# Determine the number of replicates to use for coverage calculation
nbReplicates = ifelse(length(names(table(project_Normalized$Sample))) > 5, 
                      length(names(table(project_Normalized$Sample))), 5)

# Add group coverage information to the ArchR project
project_Peaks <- addGroupCoverages(
    ArchRProj = project_Normalized, 
    maxReplicates = nbReplicates, 
    groupBy = paste("Clusters", slct.res, sep = ".")
)

### 8.2. 执行 peak calling


In [ ]:
# Find the path to MACS2
pathToMacs2 <- findMacs2()

# Perform peak calling using MACS2
project_Peaks <- addReproduciblePeakSet(
    ArchRProj = project_Peaks,
    groupBy = paste("Clusters", slct.res, sep = "."),
    pathToMacs2 = pathToMacs2
)

# Alternative peak calling method (use if MACS2 is unavailable)
# project_Peaks <- addReproduciblePeakSet(
#     ArchRProj = project_Peaks,
#     groupBy = paste("Clusters", slct.res, sep = "."),
#     peakMethod = "Tiles",
#     method = "p"
# )

# Retrieve the peak set after calling peaks
getPeakSet(project_Peaks)

# Add imputation weights to improve downstream analyses
project_Peaks <- addImputeWeights(project_Peaks)

# Save the ArchR project with peak calling results
saveArchRProject(
    ArchRProj = project_Peaks,
    outputDirectory = file.path(getwd(), "Analysis_scATACseq"), 
    load = TRUE
)

### 8.3. 识别 marker peaks

正如前面 marker genes 部分所述，ArchR 可以使用 gene、peak 或 transcription factor motif 作为特征。这里，ArchR 会在所选分辨率下识别每个 cluster 中看起来特异活跃的 peaks。


In [ ]:
# Generate a Peak Matrix for accessibility quantification
project_Peaks <- addPeakMatrix(project_Peaks)

# Identify marker peaks for each cluster using Wilcoxon test
markersPeaks <- getMarkerFeatures(
    ArchRProj = project_Peaks,
    useMatrix = "PeakMatrix",
    groupBy = paste("Clusters", slct.res, sep = "."),
    bias = c("TSSEnrichment", "log10(nFrags)"),
    testMethod = "wilcoxon"
)

# Extract significantly different peaks (FDR <= 0.01, Log2FC >= 1)
markerList <- getMarkers(markersPeaks, cutOff = "FDR <= 0.01 & Log2FC >= 1")

# View marker peaks for cluster C9
markerList[["C9"]]

为了可视化 marker genes，可以绘制热图：


In [ ]:
# Generate a heatmap of marker peaks (less stringent threshold)
heatmapPeaks <- plotMarkerHeatmap(
    seMarker = markersPeaks,
    cutOff = "FDR <= 0.1 & Log2FC >= 0.5",
    transpose = FALSE
)

# Display the heatmap
heatmapPeaks

也可以按 cluster 绘制 marker peaks 的 MA plot 和 volcano plot：


In [ ]:
# Generate MA and Volcano plots for cluster C9
map <- plotMarkers(
    seMarker = markersPeaks, 
    name = "C9",
    cutOff = "FDR <= 0.1 & Log2FC >= 1",  # Default
    plotAs = "MA"
)

vp <- plotMarkers(
    seMarker = markersPeaks, 
    name = "C9",
    cutOff = "FDR <= 0.1 & Log2FC >= 1",  # Default
    plotAs = "Volcano"
)

# Combine both plots
map | vp

还可以在 browser track 中可视化 marker peaks：


In [ ]:
# Generate a browser track visualization for CD4
plot.track1 <- plotBrowserTrack(
    ArchRProj = project_Peaks,
    groupBy = paste("Clusters", slct.res, sep = "."),
    geneSymbol = c("CD4"),
    features = getMarkers(markersPeaks, cutOff = "FDR <= 0.1 & Log2FC >= 1", returnGR = TRUE)["C9"],
    upstream = 50000, downstream = 50000
)

# Display the track
grid::grid.newpage()
grid::grid.draw(plot.track1$CD8A)

In [38]:
# Save the object and download it!!
saveRDS(project_Peaks,"project_Peaks.rds")
saveRDS(markersPeaks,"markersPeaks.rds")

## 9. Motif 富集

识别 peak sets 后，下一步是预测哪些转录因子（transcription factors, TFs）可能介导了形成这些开放染色质位点的结合事件。

ArchR 可以注释在不同细胞类型中上调或下调 peaks 富集的 TF motifs。

首先，我们基于参考数据库为 ArchRProject 添加 motif annotations，例如 CIS-BP、JASPAR、ENCODE 或 HOMER。

这里选择 CIS-BP。CIS-BP 包含来自 700 多个物种、超过 300 个 TF families 的信息，整合了 70 多个来源，其中包括 Transfac、JASPAR、Hocomoco、FactorBook、UniProbe 等数据库。

随后，我们会对显著差异 peaks 集合进行 motif enrichment 检验。


In [ ]:
# Download and reload data if needed
shell_call("gdown 1SdSmF9R3yHNWacmFf22RxpcrrKmUHM_b")
markersPeaks = readRDS("markersPeaks.rds")

shell_call("gdown 1S6fRM7_KX4kjd9ankvzA5HloJJSIM4bn")
project_Peaks = readRDS("project_Peaks.rds")

In [ ]:
## Motif Enrichment
project_Peaks <- addMotifAnnotations(ArchRProj = project_Peaks, motifSet = "cisbp", name = "Motif", force = TRUE)

# Motif enrichment in marker peaks
enrichMotifs <- peakAnnoEnrichment(
    seMarker = markersPeaks, ArchRProj = project_Peaks,
    peakAnnotation = "Motif",cutOff = "FDR <= 0.1 & Log2FC >= 0.5")        # Default

# You have two output, the Enrichment matrix and the pvalue matrix:
head(enrichMotifs@assays@data$Enrichment,10)

可以绘制热图，展示每个 cluster 中的主要 motifs。


In [ ]:
# Plot a heatmap of enriched motifs
heatmapEM <- plotEnrichHeatmap(enrichMotifs, n = 7, transpose = TRUE)
ComplexHeatmap::draw(heatmapEM, heatmap_legend_side = "bot", annotation_legend_side = "bot")

### 9.1. ChromVAR 与 motif deviation 可视化

ChromVAR 由 Greenleaf Lab 开发，用于从稀疏的染色质可及性数据中预测单细胞层面的 TF 活性富集情况。

ChromVAR 会计算：

1. “deviation”：一个经过偏差校正的度量，用于衡量某个特征（例如 motif）在单个细胞中的可及性，与基于所有细胞或样本平均水平所期望的可及性相差多远。

2. “z-score” 或 “deviation score”：对所有细胞中每个偏差校正后的 deviation 计算得到的 z-score。


In [ ]:
# Add background peaks
project_Peaks <- addBgdPeaks(project_Peaks)

# Compute deviations matrix
project_Peaks <- addDeviationsMatrix(ArchRProj = project_Peaks, peakAnnotation = "Motif",force = TRUE)

# Plot variability in motif accessibility
getVarDeviations(project_Peaks, name = "MotifMatrix", plot = TRUE)

# Save project
saveArchRProject(ArchRProj = project_Normalized,
                 outputDirectory = file.path(getwd(),"Analysis_scATACseq"))

可以展示 markers 的分布。


In [ ]:
# Define a list of motifs to analyze
motifs <- c("FOS", "JUNB")

# Retrieve motif features from the MotifMatrix that match the selected motifs
markerMotifs <- getFeatures(
  project_Peaks,
  select = paste(motifs, "_", collapse = "|", sep = ""),
  useMatrix = "MotifMatrix"
)

# Filter motif features to include only those with the "z:" prefix
markerMotifs <- grep("z:", markerMotifs, value = TRUE)

# Add imputation weights to the ArchR project for smoothing data visualization
project_Peaks <- addImputeWeights(project_Peaks)

# Generate grouped motif enrichment plots
cowp <- plotGroups(
  ArchRProj = project_Peaks,
  groupBy = paste("Clusters", slct.res, sep = "."),
  colorBy = "MotifMatrix",
  name = markerMotifs,
  imputeWeights = getImputeWeights(project_Peaks)
)

# Arrange plots in a grid layout with two columns
do.call(cowplot::plot_grid, c(list(ncol = 2), cowp))

也可以在 UMAP 中可视化 motif deviation，并观察 motif deviation 是否与对应 TF 的 gene score 相关。


In [ ]:
# Plot motif enrichment on UMAP embedding
motif.umap <- plotEmbedding(
  ArchRProj = project_Peaks,
  colorBy = "MotifMatrix",
  name = sort(markerMotifs),
  embedding = "UMAP",
  imputeWeights = getImputeWeights(project_Peaks)
)

# Display motif UMAP plots in a grid layout
do.call(cowplot::plot_grid, c(list(ncol = 2), motif.umap))

# Retrieve gene activity features related to the selected motifs
markerRNA <- getFeatures(
  project_Peaks,
  select = paste(motifs, "$", collapse = "|", sep = ""),
  useMatrix = "GeneScoreMatrix"
)

# Plot gene score matrix enrichment on UMAP embedding
gene.umap <- plotEmbedding(
  ArchRProj = project_Peaks,
  colorBy = "GeneScoreMatrix",
  name = sort(markerRNA),
  embedding = "UMAP",
  imputeWeights = getImputeWeights(project_Peaks)
)

# Display gene score UMAP plots in a grid layout
do.call(cowplot::plot_grid, c(list(ncol = 2), gene.umap))

### 9.2. clusters 之间的成对检验

可以基于两个 clusters 之间 peaks 的差异可及性，识别这两个 clusters 之间的 motif enrichment。


In [ ]:
slct.Cl1="C9"
slct.Cl2="C11"

# Perform differential analysis between C9 and C11
markerTest <- getMarkerFeatures(ArchRProj = project_Peaks,
                                useMatrix = "PeakMatrix",
                                groupBy = paste("Clusters",slct.res,sep="."),
                                testMethod = "wilcoxon",
                                bias = c("TSSEnrichment", "log10(nFrags)"),
                                useGroups = slct.Cl1, bgdGroups = slct.Cl2)

# Generate MA and Volcano plots
map.Cl1vCl2 <- markerPlot(seMarker = markerTest, name = slct.Cl1,
                        cutOff = "FDR <= 0.1 & abs(Log2FC) >= 1",
                        plotAs = "MA")
vp.Cl1vCl2 <- markerPlot(seMarker = markerTest, name = slct.Cl1,
                       cutOff = "FDR <= 0.1 & abs(Log2FC) >= 1",
                       plotAs = "Volcano")

map.Cl1vCl2 | vp.Cl1vCl2


基于 clusters 之间的成对检验，展示 motif up-enrichment 和 motif down-enrichment。


In [ ]:
# Identify significantly enriched motifs in peaks with increased accessibility
motifsUp <- peakAnnoEnrichment(ArchRProj = project_Peaks,
                               seMarker = markerTest,
                               peakAnnotation = "Motif",
                               cutOff = "FDR <= 0.1 & Log2FC >= 0.5") # Select motifs with significant FDR and Log2FC >= 0.5

# Create a data frame with motif names and -log10 adjusted p-values
df <- data.frame(TF = rownames(motifsUp), mlog10Padj = assay(motifsUp)[,1])
df <- df[order(df$mlog10Padj, decreasing = TRUE),] # Sort by significance
df$rank <- seq_len(nrow(df)) # Assign rank based on significance

# Plot enriched TF motifs with labels
ggUp <- ggplot(df, aes(rank, mlog10Padj, color = mlog10Padj)) +
    geom_point(size = 1) + ggrepel::geom_label_repel(
        data = df[rev(seq_len(30)), ], aes(x = rank, y = mlog10Padj, label = TF),
        size = 1.5, nudge_x = 2, color = "black") + theme_ArchR() +
        ylab("-log10(P-adj) Motif Enrichment") + xlab("Rank Sorted TFs Enriched") +
        scale_color_gradientn(colors = paletteContinuous(set = "comet"))

# Identify significantly enriched motifs in peaks with decreased accessibility
motifsDo <- peakAnnoEnrichment(ArchRProj = project_Peaks,
                               seMarker = markerTest,
                               peakAnnotation = "Motif",
                               cutOff = "FDR <= 0.1 & Log2FC <= -0.5") # Select motifs with Log2FC <= -0.5

df <- data.frame(TF = rownames(motifsDo), mlog10Padj = assay(motifsDo)[,1])
df <- df[order(df$mlog10Padj, decreasing = TRUE),] # Sort by significance
df$rank <- seq_len(nrow(df)) # Assign rank

# Plot TF motifs that are lost in accessibility
ggDo <- ggplot(df, aes(rank, mlog10Padj, color = mlog10Padj)) +
    geom_point(size = 1) + ggrepel::geom_label_repel(
        data = df[rev(seq_len(30)), ], aes(x = rank, y = mlog10Padj, label = TF),
        size = 1.5, nudge_x = 2, color = "black") + theme_ArchR() +
        ylab("-log10(FDR) Motif Enrichment") + xlab("Rank Sorted TFs Enriched") +
        scale_color_gradientn(colors = paletteContinuous(set = "comet"))

# Combine both plots
ggUp | ggDo

## 10. 识别正向 TF 调控因子

虽然 ATAC-seq 可以无偏地识别 TF，但从整体 motif 层面观察时，同一 TF family 中的多个 TF 往往共享相似 motif。

这使得我们难以判断具体是哪一个 TF 负责了其预测结合位点处观察到的染色质可及性变化。为了解决这个问题，ArchR 会识别这样的 TF：其基因表达水平（GeneScore）与其对应 motif 的可及性变化（使用 ChromVAR 获得的 motif deviation）呈正相关。


### 步骤 1. 识别发生偏离的 TF motifs


In [ ]:
# Extract motif deviation scores grouped by clusters
seGroupMotif <- getGroupSE(ArchRProj = project_Peaks, useMatrix = "MotifMatrix", groupBy = paste("Clusters", slct.res, sep="."))

# Extract only Z-score deviations
seZ <- seGroupMotif[rowData(seGroupMotif)$seqnames == "z",]

# Compute the maximum delta in Z-score across all clusters
rowData(seZ)$maxDelta <- lapply(seq_len(ncol(seZ)), function(x){
  rowMaxs(assay(seZ) - assay(seZ)[,x])
}) %>% Reduce("cbind", .) %>% rowMaxs

### 步骤 2. 识别相关的 TF motifs 与 TF GeneScore/表达


In [ ]:
# Compute correlations between gene scores and motif deviations
corGSM_MM <- correlateMatrices(
    ArchRProj = project_Peaks,
    useMatrix1 = "GeneScoreMatrix",
    useMatrix2 = "MotifMatrix",
    reducedDims = "Harmony" # Can also use IterativeLSI
)

# Display top correlations
head(corGSM_MM, 15)

### 步骤 3. 将最大 delta deviation 加入 correlation data frame


In [ ]:
# Annotate motifs with the max delta observed between clusters
corGSM_MM$maxDelta <- rowData(seZ)[match(corGSM_MM$MotifMatrix_name, rowData(seZ)$name), "maxDelta"]

### 步骤 4. 识别正向 TF regulators


In [ ]:
# Sort by absolute correlation and remove duplicate motifs
corGSM_MM <- corGSM_MM[order(abs(corGSM_MM$cor), decreasing = TRUE), ]
corGSM_MM <- corGSM_MM[which(!duplicated(gsub("\\-.*", "", corGSM_MM[,"MotifMatrix_name"]))), ]

# Classify TFs as positive (PLUS), negative (NEG), or neutral (NO)
corGSM_MM$TFRegulator <- "NO"
corGSM_MM$TFRegulator[which(corGSM_MM$cor > 0.1 & corGSM_MM$padj < 0.01 & corGSM_MM$maxDelta > quantile(corGSM_MM$maxDelta, 0.75))] <- "PLUS"
corGSM_MM$TFRegulator[which(corGSM_MM$cor < (-0.1) & corGSM_MM$padj < 0.01 & corGSM_MM$maxDelta > quantile(corGSM_MM$maxDelta, 0.75))] <- "NEG"

# Scatter plot of correlation vs max delta
ggplot(data.frame(corGSM_MM), aes(cor, maxDelta, color = TFRegulator)) +
  geom_point() +
  theme_ArchR() +
  geom_vline(xintercept = 0, lty = "dashed") +
  scale_color_manual(values = c("NO"="darkgrey", "PLUS"="firebrick3", "NEG"="royalblue1")) +
  xlab("Correlation To Gene Score") +
  ylab("Max TF Motif Delta") +
  scale_y_continuous(
    expand = c(0,0),
    limits = c(0, max(corGSM_MM$maxDelta)*1.05)
  )

# Display top regulators
head(as.matrix(sort(corGSM_MM[corGSM_MM$TFRegulator=="PLUS", c("GeneScoreMatrix_name", "MotifMatrix_name", "cor", "padj", "maxDelta")])), 15)
head(as.matrix(sort(corGSM_MM[corGSM_MM$TFRegulator=="NEG", c("GeneScoreMatrix_name", "MotifMatrix_name", "cor", "padj", "maxDelta")])), 5)

## 11. Co-accessibility

为了研究基因如何受到调控，例如 promoter 与 enhancer 之间的联系，ArchR 提供了 co-accessibility 分析。

Co-accessibility 指许多单细胞中两个 peaks 的可及性之间存在相关关系。换句话说，当 Peak A 在某个细胞中可及时，Peak B 往往也同时可及。

例如，co-accessibility 可以帮助可视化与某个基因 promoter 相关联的 enhancer。

说明：co-accessibility 分析可以识别细胞类型特异性 peaks。虽然这些 peaks 往往在同一细胞类型中共同可及，并且在其他细胞类型中通常都不可及，但这本身并不必然证明这些 peaks 之间存在直接调控关系。

![<i><font size=1 color="grey">from ArchR manual</font></i>](./Figures/ArchR_Coaccessibility.png){width=50% height=50%}


In [ ]:
# Add co-accessibility analysis to the ArchR project using Harmony dimensions
project_Peaks <- addCoAccessibility(ArchRProj = project_Peaks, reducedDims = "Harmony")

# Retrieve co-accessibility interactions with correlation cutoff and resolution
cA <- getCoAccessibility(
  ArchRProj = project_Peaks,
  corCutOff = 0.5,
  resolution = 10000,
  returnLoops = TRUE
)

# Display the first 10 co-accessibility interactions
head(cA$CoAccessibility, 10)

# Generate a genome browser track visualization for selected marker genes
p <- plotBrowserTrack(
  ArchRProj = project_Peaks,
  groupBy = paste("Clusters", slct.res, sep = "."),
  geneSymbol = markerGenes,
  upstream = 50000,  # Extend 50kb upstream
  downstream = 50000,  # Extend 50kb downstream
  loops = getCoAccessibility(project_Peaks)
)

# Render the genome browser plot
grid::grid.newpage()
grid::grid.draw(p$CD8A)

## 12. Footprinting

转录因子（TF）footprinting 可用于预测某个 TF 在特定位点上的精确结合位置。其原理是：TF 直接结合的 DNA 碱基会受到保护，不容易发生转座；而紧邻 TF 结合位点两侧的 DNA 碱基仍然可及。


In [ ]:
# Get motif positions
motifPositions <- getPositions(project_Peaks)

# Remove 'z:' prefix from motif names
markerMotifs <- gsub("z:", "", markerMotifs)

# Compute motif footprints
seFoot <- getFootprints(
  ArchRProj = project_Peaks,
  positions = motifPositions[markerMotifs],
  groupBy = paste("Clusters", slct.res, sep=".")
)

# Plot footprints with bias correction
plotFootprints(seFoot = seFoot,
               ArchRProj = project_Peaks,
               normMethod = "Subtract", # Options: Divide, None
               plotName = paste("Footprints-Subtract-Bias", slct.res, "cisbp", sep="."),
               addDOC = FALSE,
               smoothWindow = 5)

## 13. 轨迹分析

ArchR 可以构建一条细胞轨迹，用于近似表示从一个细胞 cluster 向另一个 cluster 的分化过程。

定义 trajectory backbone 之后，也就是一个按顺序排列的细胞群标签向量，ArchR 会为轨迹中的每个细胞识别一个 pseudo-time 值。

在结果中，ArchR 会提供 UMAP 用于可视化伪时间轨迹，也会提供热图，用于追踪 insertion/peak/gene 等信号随伪时间变化的模式。


### 13.1. 构建轨迹

首先，ArchR 会为轨迹中的每个细胞生成一个 pseudo-time 值。该值可以显示在 UMAP 上，也可以用来绘制由 spline 拟合得到的轨迹方向箭头。


In [ ]:
# Define a trajectory (e.g., C9 to C11)
trajectory <- c("C9","C11")
traj.name <- "TF.C9.C11"

# Add trajectory to the project
project_Peaks <- addTrajectory(
    ArchRProj = project_Peaks,
    name = traj.name,
    groupBy = paste("Clusters", slct.res, sep="."),
    trajectory = trajectory,
    embedding = "UMAP",
    force = TRUE
)

# Plot trajectory
plotTraj <- plotTrajectory(project_Peaks, trajectory = traj.name, colorBy = "cellColData", name = traj.name, embedding = "UMAP")
plotTraj[[1]]

### 13.2. 观察特定基因

可以可视化这条轨迹，并根据某个特定基因的 gene score 给细胞着色。


In [ ]:
# Plot trajectory of the gene CD4 using the GeneScoreMatrix, visualized in UMAP embedding
p_gene <- plotTrajectory(project_Peaks, trajectory = traj.name, colorBy = "GeneScoreMatrix", 
                         name = "CD4", continuousSet = "horizonExtra", embedding = "UMAP")

# Display the first two trajectory plots side by side
p_gene[[1]] | p_gene[[2]]

### 13.3. 伪时间热图

最后，ArchR 可以绘制热图，展示许多特征，例如 peaks、gene scores 或 motifs，在伪时间上的变化。


In [ ]:
# Obtain the gene score trajectory (normalized with log2)
trajGSM <- getTrajectory(ArchRProj = project_Peaks, name = traj.name, useMatrix = "GeneScoreMatrix", log2Norm = TRUE)

# Plot heatmap for gene score trajectory with "horizonExtra" color palette
p_trajGSM <- plotTrajectoryHeatmap(trajGSM, pal = paletteContinuous(set = "horizonExtra"))

# Generate heatmap matrix for gene score trajectory
p_trajGSM.matrix <- plotTrajectoryHeatmap(trajGSM, pal = paletteContinuous(set = "horizonExtra"), returnMatrix = TRUE)

# Obtain the peak accessibility trajectory (normalized with log2)
trajPM  <- getTrajectory(ArchRProj = project_Peaks, name = traj.name, useMatrix = "PeakMatrix", log2Norm = TRUE)

# Plot heatmap for peak accessibility trajectory with "solarExtra" color palette
p_trajPM <- plotTrajectoryHeatmap(trajPM, pal = paletteContinuous(set = "solarExtra"))

# Generate heatmap matrix for peak accessibility trajectory
p_trajPM.matrix <- plotTrajectoryHeatmap(trajPM, pal = paletteContinuous(set = "solarExtra"), returnMatrix = TRUE)

# Obtain the motif activity trajectory (without log2 normalization)
trajMM  <- getTrajectory(ArchRProj = project_Peaks, name = traj.name, useMatrix = "MotifMatrix", log2Norm = FALSE)

# Plot heatmap for motif activity trajectory with "solarExtra" color palette
p_trajMM <- plotTrajectoryHeatmap(trajMM, pal = paletteContinuous(set = "solarExtra"))

# Generate heatmap matrix for motif activity trajectory
p_trajMM.matrix <- plotTrajectoryHeatmap(trajMM, pal = paletteContinuous(set = "solarExtra"), returnMatrix = TRUE)

# Display the heatmaps
p_trajGSM
p_trajPM
p_trajMM

### 13.3. 整合式伪时间分析

如前所示，ArchR 也可以进行整合分析：结合 gene scores 和 motif accessibility 识别正向 TF，跟踪它们在伪时间上的变化，并理解它们在这条轨迹中的作用。为此，ArchR 提供 `correlateTrajectories()` 函数。该函数接受两个 `SummarizedExperiment` 对象作为输入，这两个对象来自此前运行的 `getTrajectories()` 函数。

* 步骤 1. 识别并选择 gene score 与 TF motif accessibility 相关的 motifs：


In [ ]:
# Compute correlation between gene score trajectory (trajGSM) and motif trajectory (trajMM)
# Using low stringency criteria: correlation cutoff of 0.2, variance cutoffs of 0.5 for both matrices
corGSM_MM <- correlateTrajectories(trajGSM, trajMM, 
                                   corCutOff = 0.2, varCutOff1 = 0.5, varCutOff2 = 0.5)

# Filter the gene score and motif trajectories based on correlation results
flt_trajGSM <- trajGSM[corGSM_MM[[1]]$name1, ]
flt_trajMM <- trajMM[corGSM_MM[[1]]$name2, ]

* 步骤 2. 创建一条新的轨迹，并排可视化基于 gene score 的 TF motif 与 TF motif enrichment：


In [ ]:
# Create a combined trajectory object using the filtered gene score trajectory (flt_trajGSM)
combinedTraj <- flt_trajGSM

# Normalize and combine the gene score trajectory (flt_trajGSM) and motif trajectory (flt_trajMM)
# - Scale each row (gene/motif) separately for both matrices
# - Transpose the result and add them together to integrate information from both sources
assay(combinedTraj, withDimnames=FALSE) <- t(apply(assay(flt_trajGSM), 1, scale)) + 
                                           t(apply(assay(flt_trajMM), 1, scale))

# Generate a heatmap matrix from the combined trajectory
# - returnMat = TRUE returns the matrix instead of plotting
# - varCutOff = 0 ensures no variance-based filtering
combinedMat <- plotTrajectoryHeatmap(combinedTraj, returnMat = TRUE, varCutOff = 0)

# Determine the order of rows (genes/motifs) in the combined matrix
rowOrder <- match(rownames(combinedMat), rownames(flt_trajGSM))

# Plot heatmap for the gene score trajectory, keeping row order consistent with the combined matrix
ht_GSM <- plotTrajectoryHeatmap(flt_trajGSM, pal = paletteContinuous(set = "horizonExtra"),  
                                varCutOff = 0, rowOrder = rowOrder)

# Plot heatmap for the motif trajectory, keeping row order consistent with the combined matrix
ht_MM <- plotTrajectoryHeatmap(flt_trajMM, pal = paletteContinuous(set = "solarExtra"), 
                               varCutOff = 0, rowOrder = rowOrder)

# Display both heatmaps side by side for comparison
ht_GSM + ht_MM


# 会话信息


In [ ]:
# Display session info to track package versions
sessionInfo()